In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os
from skimage import morphology, segmentation
from scipy.ndimage import rotate
from skimage import segmentation, morphology
from matplotlib.colors import hsv_to_rgb
import Chain_Analysis_Functions as caf
from scipy.ndimage import binary_opening, binary_closing

## Step 1: Read in the Image and create the initial mask

In [ ]:
original_image_name = '56-connectors.jpg'
image_path = os.path.join(os.getcwd(), '0.1_20mT', original_image_name)
original_image = Image.open(image_path).convert('RGB')
mask = caf.create_binary_mask(image_path, brightness=1.0, contrast=1.0, saturation=1.0,
                           temperature=0, R_min=0, G_min=0, B_min=30, V_min=0.1,
                           method="adaptive", adaptive_block_size=10, adaptive_offset=0.01)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 8))

ax1.imshow(mask, cmap='gray')
ax1.set_title("Original Image")
ax1.axis("off")

# Top-left corner (works as-is)
ax2.imshow(mask[0:520, 0:520], cmap='gray')
ax2.set_title("Top left corner")
ax2.axis("off")

# Bottom-right corner — corrected slicing
ax3.imshow(mask[-520:, -520:], cmap='gray')
ax3.set_title("Bottom right corner")
ax3.axis("off")

plt.tight_layout()
plt.show()

## Step 2: Rotate the mask such that the particles align into perfect rows and columns

### Perform Rough Rotation

In [ ]:
angle = -1
rot_mask = caf.rotate(mask, angle, reshape=False, order=0, mode='constant', cval=0)
caf.display_mask(rot_mask, 20, 20)

#### Identify the appropriate rotation angle

In [ ]:
rot_angle_mask = morphology.remove_small_objects(rot_mask, min_size= 100) #Remove noise to keep it from interfering
caf.display_mask(rot_angle_mask,10,10)
rot_angle_mask, angle = caf.rotate_mask_until_balanced(rot_angle_mask, angle_step=0.1, tolerance=0.001, min_step=0.001, max_angle=5, search_rows = 100, search_columns = 600)
print(angle)

In [ ]:
angle = 0.682763671875

In [ ]:
mask_rotated = caf.rotate(rot_mask, angle, reshape=False, order=0, mode='constant', cval=0)
caf.display_mask(mask_rotated)

In [ ]:
mask = mask_rotated
del mask_rotated

## Step 3: Identify the reference particle bounding box

In [ ]:
ref_mask = mask[471:774, 854:1159]
print(np.shape(ref_mask))
caf.display_mask(ref_mask, 10, 10)

In [ ]:
ref_bbox = (471,854,774,1159)
caf.show_reference(mask, ref_bbox)

## Step 4: Ensure reference crop is filled properly with set parameters

### Perform Morphological Operations manually

In [ ]:
# Define the minimum connected-component size retained during small-object removal.
min_size = 20

# Define the radius of the disk-shaped structuring element used for the initial morphological opening.
opening_disk_size = 2

# Define the number of pixels added as padding around the reference mask.
pad = 100

# Define the radius of the disk-shaped structuring element used for morphological closing.
closing_disk_size1 = 35

# Define the minimum object size intended for retaining larger connected components.
larger_object_size = 200

# Define the maximum horizontal gap filled in the upper and lower portions of the reference mask.
max_row_gap1 = 150

# Define the maximum horizontal gap filled across the padded reference mask.
max_row_gap = 50

# Define the maximum vertical gap filled across the padded reference mask.
max_column_gap = 50

# Define the radius of the disk-shaped structuring element used for the final morphological opening.
opening_disk_size2 = 10

# Define a secondary maximum horizontal gap size for additional row-gap processing.
max_row_gap2 = 5

# Define a secondary maximum vertical gap size for additional column-gap processing.
max_column_gap2 = 4

# Unpack the reference particle bounding box into minimum and maximum row and column coordinates.
minr, minc, maxr, maxc = ref_bbox

# Extract the reference particle from the full mask using the specified bounding box
# and convert the result to a Boolean mask.
ref_crop = mask[minr:maxr, minc:maxc].astype(bool)

# Create an independent copy of the original cropped reference mask for later comparison.
cropmask = ref_crop.copy()

# Display the initially cropped reference mask for visual inspection.
caf.display_mask(ref_crop,5,5)

# Remove connected components smaller than the specified minimum size.
ref_crop = morphology.remove_small_objects(ref_crop, min_size=min_size)

# Display the mask after small-object removal.
#caf.display_mask(ref_crop,5,5)

# Apply binary morphological opening using a disk-shaped structuring element
# to remove small protrusions and smooth the mask.
ref_crop = binary_opening(ref_crop, structure = morphology.disk(opening_disk_size))

# Remove a manually selected region from the upper/lower-left portion of the reference mask.
ref_crop[150:170,:50] = 0

# Remove another manually selected region from the upper-left portion of the reference mask.
ref_crop[50:70,:20] = 0

# Remove a manually selected region from the right side of the reference mask.
ref_crop[68:140,-40:] = 0

# Remove a small manually selected region from the lower-middle portion of the reference mask.
ref_crop[232:240,30:40] = 0

# Remove a manually selected region from the lower-right portion of the reference mask.
ref_crop[150:250,-30:] = 0

# Remove another manually selected region from the lower-right edge of the reference mask.
ref_crop[250:260,-20:] = 0

# Remove another manually selected region from the right side of the reference mask.
ref_crop[67:100,-35:] = 0

# Display the reference mask after the initial morphological processing
# and manually selected region removals.
caf.display_mask(ref_crop,5,5)

# Fill horizontal gaps up to max_row_gap1 pixels wide within the top 30 rows of the reference mask.
ref_crop[0:30,:] = caf.fill_row_gaps2(ref_crop[0:30,:], max_gap = max_row_gap1)

# Fill horizontal gaps up to max_row_gap1 pixels wide within the bottom 30 rows of the reference mask.
ref_crop[-30:,:] = caf.fill_row_gaps2(ref_crop[-30:,:], max_gap = max_row_gap1)

# Display the reference mask after filling gaps near its upper and lower boundaries.
caf.display_mask(ref_crop,5,5)

# Add zero-valued padding around the reference mask to provide additional space
# for subsequent morphological operations.
ref_crop = np.pad(ref_crop, pad_width = pad, mode='constant')

# Display the padded reference mask.
#caf.display_mask(ref_crop,5,5)

# Fill horizontal gaps up to max_row_gap pixels wide across the padded reference mask.
ref_crop = caf.fill_row_gaps2(ref_crop, max_gap = max_row_gap)

# Display the reference mask after filling horizontal gaps.
#caf.display_mask(ref_crop,5,5)

# Fill vertical gaps up to max_column_gap pixels wide across the padded reference mask.
ref_crop = caf.fill_column_gaps(ref_crop, max_gap = max_column_gap)

# Display the reference mask after filling vertical gaps.
caf.display_mask(ref_crop, 5, 5)

# Apply binary morphological closing with a disk-shaped structuring element
# to close gaps and connect nearby regions within the reference mask.
ref_crop = binary_closing(ref_crop, structure = morphology.disk(closing_disk_size1))

# Display the reference mask after morphological closing.
caf.display_mask(ref_crop, 5, 5)

# Apply binary morphological opening with a disk-shaped structuring element
# to remove small protrusions and smooth the result after closing.
ref_crop = binary_opening(ref_crop, structure = morphology.disk(opening_disk_size2))

# Display the reference mask after the final morphological opening.
caf.display_mask(ref_crop, 5, 5)

# Remove the temporary padding from all four sides of the processed reference mask.
ref_crop = ref_crop[pad:-pad, pad:-pad]

# Convert the processed binary reference mask into a three-channel floating-point image
# so that the original mask can be displayed as a colored overlay.
overlay_img = np.stack([ref_crop]*3, axis=-1).astype(float)

# Overlay the original cropped mask in red (R=1, G=0, B=0).
overlay_img[cropmask.astype(bool), 0] = 1.0  # Red

# Set the green channel to zero wherever the original cropped mask is present.
overlay_img[cropmask.astype(bool), 1] = 0.0  # Green

# Set the blue channel to zero wherever the original cropped mask is present.
overlay_img[cropmask.astype(bool), 2] = 0.0  # Blue

# Create a figure for displaying the processed mask and original mask overlay.
plt.figure(figsize=(6, 6))

# Display the processed reference mask with the original cropped mask shown in red.
plt.imshow(overlay_img)

# Set the title for the overlay visualization.
plt.title("Overlay Mask in Red")

# Remove the plot axes for a cleaner image display.
plt.axis('off')

# Display the final overlay visualization.
plt.show()


## Step 5: Ensure Last Data is Properly Identified

In [ ]:
last_x, last_y = caf.check_last_row_and_column(mask, min_size = 100, last_row = 200, last_column = 200, plot = True)

In [ ]:
minr = 70
maxr = 373
minc = 57
maxc = 362
ref_bbox = (minr,minc,maxr,maxc)
print(maxr-minr,maxc-minc)
caf.show_reference(mask, ref_bbox)

## Step 6: Ensure Array and Partiles are Properly Captured

### Check X

In [ ]:
dx_offset = 169
stagger_x = False
stagger_x_frequency = 2

checking = True
num_rows = 1
num_cols = 15
particle_mask, debris_mask = caf.extract_particles_and_debris(mask, ref_bbox, ref_crop, min_size = 100, pad=10, max_gap = 10, dx_offset = dx_offset, dy_offset = 0,
                                                         stagger_x = stagger_x, stagger_y = False, stagger_y_frequency = 2, erode_pixels = 0, checking = checking,
                                                         check_last_row =  200, check_last_column = 200, num_rows = num_rows, num_cols = num_cols)

### Check Y

In [ ]:
dy_offset = 186
stagger_y = True
stagger_y_frequency = 3

checking = True
num_rows = 14
num_cols = 1
particle_mask, debris_mask = caf.extract_particles_and_debris(mask, ref_bbox, ref_crop, min_size = 100, pad=10, max_gap = 10, dx_offset = dx_offset, dy_offset = dy_offset,
                                                         stagger_x = stagger_x, stagger_y = stagger_y, stagger_y_frequency = stagger_y_frequency, erode_pixels = 0, checking = checking,
                                                         check_last_row =  200, check_last_column = 200, num_rows = num_rows, num_cols = num_cols)

## Step 7: Apply all parameters and pull out particles from debris

In [ ]:
dx_offset = 169
stagger_x = False
stagger_x_frequency = 2

dy_offset = 186
stagger_y = True
stagger_y_frequency = 3

checking = False
erode_pixels = 2
num_rows = 14
num_cols = 15
particle_mask, debris_mask = caf.extract_particles_and_debris(mask, ref_bbox, ref_crop, min_size = 100, pad=10, max_gap = 10, dx_offset = dx_offset, dy_offset = dy_offset,
                                                          stagger_x = stagger_x, stagger_y = stagger_y, stagger_x_frequency = stagger_x_frequency, stagger_y_frequency = stagger_y_frequency,
                                                          erode_pixels = erode_pixels, checking = checking, check_last_row =  200, check_last_column = 200, num_rows = num_rows, num_cols = num_cols)

In [ ]:
caf.display_mask(particle_mask, 10, 10)

In [ ]:
caf.display_mask(debris_mask,10,10)

In [ ]:
caf.save_mask(particle_mask, image_path, '_0.png')
caf.save_mask(debris_mask, image_path,'_debris_0.png')

## Step 7: Identify bounds where particles are missing on the final wafer

### Step 7a: Load in cut wafer image and convert to a mask

In [ ]:
original_image_name = '0.1wt_strong_al_n1_VSM.jpg'
wafer_path = os.path.join(os.getcwd(), '0.1_20mT', original_image_name)
cropped_image = caf.show_original_image(wafer_path, x_size = 10, y_size = 10)

In [ ]:
cut_mask = caf.cropped_image_to_mask(cropped_image, method="otsu")

### Step 7b: Perform Initial Crop

In [ ]:
ymin = 20
ymax = 3400
xmin = 100
xmax = 2280
cut_mask_new = cut_mask[ymin:ymax, xmin:xmax]
caf.display_mask(cut_mask_new,10,10)

### Step 7c: Finely Rotate the Mask

In [ ]:
rot_angle_mask = ~cut_mask_new
rot_angle_cut_mask = morphology.remove_small_objects(rot_angle_mask, min_size= 200) #Remove noise to keep it from interfering
rot_angle_cut_mask = rotate(rot_angle_cut_mask, .25780185, reshape=False, order=0, mode='constant', cval=0)
search_columns = 300
search_rows = 20
check_mask = rot_angle_cut_mask[295:,100:]
ncols = check_mask.shape[1]
av1 = caf.first_nonzero_average(check_mask, range(0, search_columns),search_rows)
av2 = caf.first_nonzero_average(check_mask, range(ncols-search_columns, ncols), search_rows)
diff = av1 - av2
print(diff)
caf.display_mask(check_mask, 10,10)

In [ ]:
angle = .25780185

In [ ]:
cut_mask_rotated = rotate(cut_mask_new, angle, reshape=False, order=0, mode='constant', cval=0)
caf.display_mask(cut_mask_rotated, 10, 10)

### Step 7d: Identify the appropriate bounds in particle mask

In [ ]:
match_mask = particle_mask.copy()
modify_rows_left = 37
modify_rows_right = -4
modify_cols_top = 7
modify_cols_bottom = -20
cut_mask2 =  caf.match_masks(~cut_mask_rotated, match_mask, modify_rows_left, modify_rows_right, modify_cols_top, modify_cols_bottom)

In [ ]:
print(np.shape(cut_mask2))
caf.display_mask(cut_mask2, 10, 10)

### Step 7e: Add Rows and Columns to Match the Particle Mask

In [ ]:
cut_mask3 = caf.add_rows_to_match(~cut_mask2, particle_mask)

### Step 7g: Clean the mask to separate the area that has no particles

In [ ]:
caf.display_mask(cut_mask3.astype(np.uint8), 10, 10)
cut_mask5 = binary_closing(cut_mask3,structure = morphology.disk(20))
caf.display_mask(cut_mask5.astype(np.uint8), 10, 10)
cut_mask6 = caf.fill_small_holes2(cut_mask5, 50000, connectivity=1)
caf.display_mask(cut_mask6.astype(np.uint8), 10, 10)
cut_mask7 = ~caf.fill_column_gaps(~cut_mask6,30)
caf.display_mask(cut_mask7.astype(np.uint8), 10, 10)
cut_mask8= cut_mask7.copy()
cut_mask8 = caf.fill_row_gaps2(cut_mask8,100)
cut_mask8[0:3000,:] = caf.fill_row_gaps2(cut_mask8[0:3000,:],100)
cut_mask8[3000:,:] = caf.fill_row_gaps2(cut_mask8[3000:,:],50)
caf.display_mask(cut_mask8,10,10)
cut_mask9= cut_mask8.copy()
cut_mask9[5000:5280,2500:3500] = 1
caf.display_mask(cut_mask9,10,10)
cut_mask9[:3000,:] = caf.fill_column_gaps(cut_mask9[:3000,:],100)
caf.display_mask(cut_mask9,10,10)
cut_mask9[3000:,:3600] = caf.fill_column_gaps(cut_mask9[3000:,:3600],30)
caf.display_mask(cut_mask9,10,10)
cut_mask9[3000:,3600:] = caf.fill_column_gaps(cut_mask9[3000:,3600:],50)
caf.display_mask(cut_mask9,10,10)
cut_mask10 = ~cut_mask9
cut_mask10[:,2500:] = caf.fill_row_gaps2(cut_mask10[:,2500:],500)
cut_mask10[2500:,:] = caf.fill_row_gaps2(cut_mask10[2500:,:],500)
cut_mask10 = caf.fill_column_gaps(cut_mask10,700)
caf.display_mask(cut_mask10,10,10)
outline = caf.mask_outline(cut_mask10)
caf.display_mask(outline,10,10)
cut_mask11 = cut_mask3 | cut_mask10
caf.display_mask(cut_mask11,10,10)
cut_mask12 = cut_mask11 &~outline
caf.display_mask(cut_mask12,10,10)
cut_mask13 = cut_mask12.copy()
cut_mask13[:5100,:] = caf.row_threshold_mask(cut_mask13[:5100,:],5550)
caf.display_mask(cut_mask13,10,10)
cut_mask13[5100:,:] = caf.row_threshold_mask(cut_mask13[5100:,:],5800)
caf.display_mask(cut_mask13,10,10)
cut_mask13 = caf.column_threshold_mask(cut_mask13,5400)
caf.display_mask(cut_mask13,10,10)
cut_mask14 = ~cut_mask13.astype(bool)
cut_mask14[:,:200] = 0
cut_mask14[:2200,:350] = 0
cut_mask14[:40,:] = 0
caf.display_mask(cut_mask14,10,10)
cut_mask14[:5000,:] = morphology.remove_small_objects(cut_mask14[:5000,:].astype(bool), min_size=10000)
cut_mask14[5000:,3600:] = morphology.remove_small_objects(cut_mask14[5000:,3600:].astype(bool), min_size=10000)
caf.display_mask(cut_mask14,10,10)
cut_mask14[:,:400] = caf.fill_column_gaps(cut_mask14[:,:400],400)
caf.display_mask(cut_mask14,10,10)
cut_mask14 = ~cut_mask14
caf.display_mask(cut_mask14,10,10)
cut_mask15 = morphology.remove_small_objects(cut_mask14.astype(bool), min_size=5000000)
caf.display_mask(cut_mask15,10,10)
plot = True
small_object_size = 2000
max_column_gap1 = 50
mac_column_gap2 = 350
cut_mask18 = ~cut_mask15
if plot:
    print('Invert Mask')
    caf.display_mask(cut_mask18.astype(np.uint8), 10, 10)

cut_mask18 = morphology.remove_small_objects(cut_mask18, min_size=small_object_size)
if plot:
    print('Remove tiny specs')
    caf.display_mask(cut_mask18.astype(np.uint8), 10, 10)
max_column_gap1 = 50
mac_column_gap2 = 350
cut_mask19 = cut_mask18.copy()
cut_mask19[100:-100,:] = caf.fill_column_gaps(cut_mask18[100:-100,:], max_gap = max_column_gap1)
caf.display_mask(cut_mask19.astype(np.uint8), 10, 10)
pad = 500
cut_mask20 = np.pad(cut_mask19,pad)
caf.display_mask(cut_mask20.astype(np.uint8), 10, 10)
cut_mask20 = caf.fill_column_gaps(cut_mask20, max_gap = 250)
caf.display_mask(cut_mask20.astype(np.uint8), 10, 10)
cut_mask21 = caf.fill_row_gaps2(cut_mask20, max_gap = 300)
cut_mask21 = cut_mask21[pad:-pad,pad:-pad]
caf.display_mask(cut_mask21.astype(np.uint8), 10, 10)


## Step 8: Filter particle mask using blank space from cut mask

In [ ]:
original_image_name = '0.1wt_strong_al_n1_VSM.jpg'
wafer_path = os.path.join(os.getcwd(), '0.1_20mT', original_image_name)
cropped_image = caf.show_original_image(wafer_path, x_size = 10, y_size = 10)

In [ ]:
filtered_particle_mask = particle_mask & cut_mask21
caf.display_mask(filtered_particle_mask,10,10)

In [ ]:
filtered_debris_mask = debris_mask & cut_mask21
caf.display_mask(filtered_debris_mask,10,10)

In [ ]:
caf.save_mask(filtered_particle_mask,image_path,'_3.png')
caf.save_mask(filtered_debris_mask,image_path,'_debris_3.png')

# Identifying Chains in the Mask

## Below is a limited sample of the code that identifies all of the different chains within the mask. The full code can be run in "Run_chain_analysis.py"

In [ ]:
original_image_name = '56-connectors.jpg'
image_path = os.path.join(os.getcwd(), '0.1_20mT', original_image_name)
original_image = Image.open(image_path).convert('RGB')
mask_name = '56-connectors_cleaned_mask.png'
filtered_particle_mask_name = os.path.join(os.getcwd(), '0.1_20mT', mask_name)
filtered_particle_mask = Image.open(filtered_particle_mask_name).convert('L')
filtered_particle_mask = np.array(filtered_particle_mask) == 255
caf.display_mask(filtered_particle_mask)

In [ ]:
particle_bounds = (800,1200,800,1200)
region_counter, chain_mask, particle_region_masks = caf.label_mask_16(filtered_particle_mask, original_image, particle_bounds, disk_size=0, connectivity=1,
                                                                    branch_length_fraction=0.007, global_min_branch_length=2, min_region_size=200, debug_plots = False,
                                                                    prune=True, prune_branch_length=5, max_hole_size=20, vertical_prune_length = 4)

## Load Completed Labelling for a Wafer

In [ ]:
mask_name = '56-labels.csv'
wafer_path = os.path.join(os.getcwd(), '0.1_20mT', mask_name)
full_chain_mask = np.loadtxt(wafer_path, delimiter=",")
chain_mask = full_chain_mask[800:1600, 800:1600]
unique_labels = np.unique(chain_mask)
unique_labels = unique_labels[unique_labels != 0]
num_labels = len(unique_labels)

# Generate N visually distinct colors using HSV space
hues = np.linspace(0, 1, num_labels, endpoint=False)
np.random.seed(np.random.randint(0,100))  # Optional: fix randomness
np.random.shuffle(hues)  # Shuffle to avoid nearby labels looking similar
colors = hsv_to_rgb(np.stack([hues, np.ones_like(hues)*0.65, np.ones_like(hues)*0.95], axis=1))

# Create a mapping from label to color
label_to_color = {label: np.append(colors[i], 1.0) for i, label in enumerate(unique_labels)}  # RGBA

# Create the overlay image
overlay_img = np.zeros((*chain_mask.shape, 4), dtype=float)
for label, rgba in label_to_color.items():
    overlay_img[chain_mask == label] = rgba

# Add black boundaries
boundaries = segmentation.find_boundaries(chain_mask.astype(np.int32), mode='outer')
overlay_img[boundaries] = [0, 0, 0, 1]

# Display the result
fig, ax1 = plt.subplots(1, 1, figsize=(20, 10))
ax1.imshow(overlay_img)
ax1.set_title('Watershed-filled branches (globally unique colors) with black outlines')
ax1.axis('off')
plt.show()

# Analyzing Chain Data

### Analyze the properties of a small subset of the identified chains and debris in the image

In [ ]:
mask_name = '56-connectors_cleaned_mask_debris.png'
debris_mask_name = os.path.join(os.getcwd(), '0.1_20mT', mask_name)
debris_mask = Image.open(debris_mask_name).convert('L')
debris_mask = np.array(debris_mask) == 255
debris_mask = debris_mask[0:500,0:500] #Look at a small number of particles at once

In [ ]:
df = caf.analyze_clusters(chain_mask, 1, angle_offset = None, fixed_angle = 0, plot = False, print_statement = False)

In [ ]:
ex_mask = np.where(chain_mask == 21881, 5, 0)
caf.display_mask(ex_mask, 10, 10)
ex_df = caf.analyze_clusters(ex_mask, 1, angle_offset = None, fixed_angle = 0, plot = True, print_statement = True)